# 🚰 Water Pump Challenge - Getting Started

## Description du Problème

La Tanzanie possède des milliers de points d'eau (pompes à eau) répartis sur tout son territoire. Ces infrastructures sont essentielles pour fournir de l'eau potable aux populations locales. Cependant, maintenir ces pompes en état de fonctionnement est un défi logistique majeur.

### Objectif

**Votre mission** : Construire un modèle de machine learning capable de prédire l'état de fonctionnement d'une pompe à eau en fonction de ses caractéristiques.

### Les Classes à Prédire

Chaque pompe peut être dans l'un des trois états suivants :

| Classe | Description |
|--------|-------------|
| `functional` | La pompe est opérationnelle, aucune réparation nécessaire |
| `functional needs repair` | La pompe fonctionne mais nécessite des réparations |
| `non functional` | La pompe n'est pas opérationnelle |

### Métrique d'Évaluation

Vos prédictions seront évaluées avec le **taux de classification** (accuracy) :

$$\text{Accuracy} = \frac{\text{Nombre de prédictions correctes}}{\text{Nombre total de prédictions}}$$

---

## 1. Chargement des Données

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuration de l'affichage
pd.set_option('display.max_columns', 50)
plt.style.use('seaborn-v0_8-whitegrid')

In [ ]:
# Charger les données d'entraînement
train_features = pd.read_csv('data/train_features.csv')
train_labels = pd.read_csv('data/train_labels.csv')

# Charger les données de test (à prédire)
test_features = pd.read_csv('data/test_features.csv')

print(f"Données d'entraînement : {train_features.shape[0]} échantillons, {train_features.shape[1]} features")
print(f"Données de test : {test_features.shape[0]} échantillons à prédire")

In [ ]:
# Aperçu des features
train_features.head()

In [ ]:
# Aperçu des labels
train_labels.head()

## 2. Description des Features

Voici la liste des variables disponibles :

| Variable | Description |
|----------|-------------|
| `id` | Identifiant unique de la pompe |
| `amount_tsh` | Quantité totale d'eau disponible (Total Static Head) |
| `date_recorded` | Date d'enregistrement des données |
| `funder` | Organisme ayant financé le puits |
| `gps_height` | Altitude du puits (mètres) |
| `installer` | Organisation ayant installé le puits |
| `longitude` | Coordonnée GPS (longitude) |
| `latitude` | Coordonnée GPS (latitude) |
| `wpt_name` | Nom du point d'eau |
| `basin` | Bassin géographique |
| `region` | Région administrative |
| `population` | Population autour du puits |
| `construction_year` | Année de construction |
| `extraction_type` | Type d'extraction de l'eau |
| `management` | Mode de gestion |
| `payment` | Type de paiement pour l'eau |
| `water_quality` | Qualité de l'eau |
| `quantity` | Quantité d'eau disponible |
| `source` | Source de l'eau |
| `waterpoint_type` | Type de point d'eau |

## 3. Exploration des Données

In [ ]:
# Distribution des classes
class_distribution = train_labels['status_group'].value_counts()
print("Distribution des classes :")
print(class_distribution)
print(f"\nPourcentages :")
print(train_labels['status_group'].value_counts(normalize=True) * 100)

In [ ]:
# Visualisation de la distribution des classes
plt.figure(figsize=(8, 5))
colors = ['#2ecc71', '#f39c12', '#e74c3c']
class_distribution.plot(kind='bar', color=colors)
plt.title('Distribution des états des pompes à eau')
plt.xlabel('État')
plt.ylabel('Nombre de pompes')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Types de données
print("Types de données :")
print(train_features.dtypes.value_counts())

In [ ]:
# Valeurs manquantes
missing = train_features.isnull().sum()
missing_pct = (missing / len(train_features)) * 100
missing_df = pd.DataFrame({'Manquantes': missing, 'Pourcentage': missing_pct})
print("Valeurs manquantes :")
print(missing_df[missing_df['Manquantes'] > 0].sort_values('Pourcentage', ascending=False))

## 4. Exemple de Modèle Baseline

Voici un exemple simple pour vous aider à démarrer. Ce modèle utilise uniquement quelques features numériques et un Random Forest basique.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

# Fusionner features et labels pour l'entraînement
train_df = train_features.merge(train_labels, on='id')

# Sélectionner quelques features numériques simples
numeric_features = ['amount_tsh', 'gps_height', 'longitude', 'latitude', 
                    'population', 'construction_year']

X = train_df[numeric_features].fillna(0)
y = train_df['status_group']

# Modèle baseline
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# Validation croisée
scores = cross_val_score(model, X, y, cv=5, scoring='accuracy')
print(f"Accuracy en validation croisée : {scores.mean():.4f} (+/- {scores.std():.4f})")

## 5. Créer une Soumission

Pour soumettre vos prédictions, vous devez créer un fichier CSV avec deux colonnes :
- `id` : l'identifiant de la pompe
- `status_group` : votre prédiction (`functional`, `functional needs repair`, ou `non functional`)

In [ ]:
# Entraîner le modèle sur toutes les données d'entraînement
model.fit(X, y)

# Préparer les données de test
X_test = test_features[numeric_features].fillna(0)

# Faire les prédictions
predictions = model.predict(X_test)

# Créer le fichier de soumission
submission = pd.DataFrame({
    'id': test_features['id'],
    'status_group': predictions
})

# Sauvegarder
submission.to_csv('ma_soumission.csv', index=False)
print("Fichier de soumission créé : ma_soumission.csv")
print(f"\nAperçu des prédictions :")
print(submission.head(10))

In [ ]:
# Distribution des prédictions
print("Distribution de vos prédictions :")
print(submission['status_group'].value_counts())

## 6. Évaluer votre Soumission

Une fois votre fichier `ma_soumission.csv` créé, utilisez le notebook `evaluate_submission.ipynb` pour obtenir votre score.

---

## 🎯 Pistes d'Amélioration

Voici quelques idées pour améliorer votre score :

1. **Feature Engineering** :
   - Encoder les variables catégorielles (LabelEncoder, OneHotEncoder)
   - Créer de nouvelles features (âge de la pompe, etc.)
   - Gérer les valeurs manquantes de manière plus intelligente

2. **Exploration des données** :
   - Analyser la corrélation entre les features et la target
   - Identifier les features les plus importantes
   - Visualiser les distributions par classe

3. **Modèles** :
   - Essayer différents algorithmes (XGBoost, LightGBM, etc.)
   - Optimiser les hyperparamètres
   - Ensemble de modèles

4. **Validation** :
   - Utiliser une validation croisée stratifiée
   - Attention au déséquilibre des classes

**Bon courage ! 🚀**